## Evaluasi Kinerja Model TF-IDF dan RNN pada Sistem Klasifikasi Berita Hoaks Politik Berbahasa Indonesia

## 1. Data Preprocessing

### 1.1 Import the Libraries

In [1]:
import pandas as pd
import pickle
import os

from pathlib import Path
from sklearn.model_selection import train_test_split

### 1.1 Define Political Keyword

In [2]:
POLITIK_KEYWORDS = [
    "politik", "pemilu", "pemilihan umum", "pilkada", "pileg", "pilpres", "kampanye",
    "partai", "partai politik", "caleg", "capres", "cawapres", "calon presiden",
    "calon wakil presiden", "calon legislatif", "suara", "TPS", "DPT", "KPU", "Bawaslu",
    "politikus", "politisi", "anggota DPR", "DPR", "DPRD", "MPR", "lembaga legislatif",
    "parlemen", "ormas", "LSM", "koalisi", "oposisi", "kubu", "kabinet", "menteri",
    "reshuffle", "pemerintahan", "kekuasaan", "pemangku kebijakan", "kepala daerah",
    "gubernur", "bupati", "wali kota", "presiden", "wakil presiden", "sidang paripurna",
    "perppu", "peraturan", "undang-undang", "RUU", "RUU KUHP", "UU ITE", "konstitusi",
    "amandemen", "demokrasi", "otoriter", "otoritarian", "diktator", "sistem politik",
    "ideologi", "pancasila", "reformasi", "orba", "orde baru", "orde lama", "kebijakan",
    "anggaran", "APBN", "APBD", "korupsi", "nepotisme", "kolusi", "KKN", "KPK", "MK",
    "MA", "hukum tata negara", "pelanggaran HAM", "demonstrasi", "aksi", "unjuk rasa",
    "aktivis", "suara rakyat", "politik identitas", "politik uang", "black campaign",
    "hoaks politik", "buzzer", "opini publik"
]

POLITIK_SET = set(POLITIK_KEYWORDS)

### 1.2 Utility Functions (Clean, Stopwords, Detector)

In [3]:
def load_stopwords(path):
    with open(path, "r", encoding="utf-8") as f:
        return set([w.strip() for w in f.readlines() if w.strip()])

def clean_text(text):
    return text.lower()

def is_politik(text):
    if pd.isna(text) or not text:
        return 0
    tokens = set(text.split())
    return int(len(tokens & POLITIK_SET) > 0)

#### 1.4 Load and Clean Each Dataset

In [4]:
base_path = Path.home() / "Work/GitHub/Personal/Project/Training/dataset/raw"

dataset_files = [
    "dataset_cnn_10k_cleaned.csv",
    "dataset_kompas_4k_cleaned.csv",
    "dataset_tempo_6k_cleaned.csv",
    "dataset_turnbackhoax_10_cleaned.csv"
]

paths = [base_path / f for f in dataset_files]

stopwords_path = base_path / "stopwords_id.txt"
stopwords = load_stopwords(stopwords_path) 
all_dfs = []

TEXT_CANDIDATES = ["text_new", "Clean Narasi", "FullText", "Narasi"]
LABEL_CANDIDATES = ["hoax", "label"]

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

for path in paths:
    df = pd.read_csv(str(path))
    
    text_col = pick_col(df, TEXT_CANDIDATES)
    label_col = pick_col(df, LABEL_CANDIDATES)

    print(f"Loaded: {path} | Rows: {len(df)}")
    print("Detected text column:", text_col)
    print("Detected label column:", label_col)

    df = df[[text_col, label_col]].rename(columns={text_col: 'text_raw', label_col: 'label'})
    
    # Cleaning
    df['text_clean'] = df['text_raw'].astype(str).apply(clean_text)
    df['text_clean'] = df['text_clean'].apply(
        lambda x: " ".join([w for w in x.split() if w not in stopwords])
    )

    all_dfs.append(df)

Loaded: /home/calista/Work/GitHub/Personal/Project/Training/dataset/raw/dataset_cnn_10k_cleaned.csv | Rows: 9630
Detected text column: text_new
Detected label column: hoax
Loaded: /home/calista/Work/GitHub/Personal/Project/Training/dataset/raw/dataset_kompas_4k_cleaned.csv | Rows: 4750
Detected text column: text_new
Detected label column: hoax
Loaded: /home/calista/Work/GitHub/Personal/Project/Training/dataset/raw/dataset_tempo_6k_cleaned.csv | Rows: 6592
Detected text column: text_new
Detected label column: hoax
Loaded: /home/calista/Work/GitHub/Personal/Project/Training/dataset/raw/dataset_turnbackhoax_10_cleaned.csv | Rows: 10381
Detected text column: Clean Narasi
Detected label column: hoax


#### 1.5 Dataset Consolidation

In [5]:
df_gabungan = pd.concat(all_dfs, ignore_index=True)
print("Total setelah concat:", len(df_gabungan))

Total setelah concat: 31353


#### 1.6 Filter Data Politic

In [6]:
df_gabungan['is_politik'] = df_gabungan['text_clean'].apply(is_politik)

df_politik = df_gabungan[df_gabungan['is_politik'] == 1].copy()
print("Total politik only:", len(df_politik))

if df_politik.empty:
    raise ValueError("Tidak ada data politik.")

Total politik only: 20599


#### 1.7 Dataset Final

In [7]:
df_final = df_politik.copy()

print(df_final['label'].value_counts())
print("Total data (imbalanced):", len(df_final))

label
0    19020
1     1579
Name: count, dtype: int64
Total data (imbalanced): 20599


### 1.8 Train Test Split

In [8]:
X_all = df_final['text_clean'].tolist()
y_all = df_final['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print("Train:", len(X_train))
print("Test :", len(X_test))

Train: 16479
Test : 4120


### 1.9 Save Processed the Dataset

In [9]:
pickle.dump(X_train, open("../dataset/processed/01_after_preprocessing/X_train.pkl", "wb"))
pickle.dump(X_test,  open("../dataset/processed/01_after_preprocessing/X_test.pkl", "wb"))
pickle.dump(y_train, open("../dataset/processed/01_after_preprocessing/y_train.pkl", "wb"))
pickle.dump(y_test,  open("../dataset/processed/01_after_preprocessing/y_test.pkl", "wb"))

df_final.to_csv("../dataset/processed/01_after_preprocessing/df_final.csv", index=False)